# Routiq AI Coach — interactive notebook v0.6

Select **.venv (Python 3.12)** and click **Restart**, then **Run All**.

The notebook starts offline. At the **You:** box:

- `/live`: enable real OpenAI replies after confirming `yes` (or `enable live`).
- `/offline`: return to offline mode.
- `/help`, `/samples`, `/status`, `/reset`: inspect and control your session.
- `exit`: stop and disable live mode.

Before `/live`, create **.env inside routiq-ai-coach** from `.env.example` and fill in `OPENAI_API_KEY` and `OPENAI_MODEL`. Do not paste your key into chat or notebook cells. No model is selected automatically. The parent project's .env is not used.

**Live mode sends questions to OpenAI and may incur charges.** It supports general habit coaching, with a CrewAI Flow and Habit Coach agent. Personal records, progress analysis, habit modifications and saved conversation memory are not connected. Each question is independent. Authentication here is synthetic and scoped to this in-memory demonstration; production authentication remains unavailable.

Configuration, client and chat cells initialize their own prerequisites. Rerunning the chat cell resets history and returns to offline mode. Clear outputs before sharing.


In [ ]:
# 1. Find the existing module and verify its local kernel. No user/drive hardcoding.
from pathlib import Path
import platform
import sys

start = Path.cwd().resolve()
candidates = [Path(sys.prefix).resolve().parent, start, *start.parents,
              start / "routiq-ai-coach",
              start / "projects" / "routiq" / "routiq-ai-coach"]
project_root = next((p for p in candidates
                     if (p / "pyproject.toml").is_file()
                     and (p / "app" / "config" / "routiq_chatbot.json").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the Routiq AI Coach module and select its .venv kernel.")
project_root = project_root.resolve()
expected_venv = (project_root / ".venv").resolve()
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use the project's Python 3.12 kernel; supported range is >=3.10,<3.14.")
if Path(sys.prefix).resolve() != expected_venv:
    raise RuntimeError("Wrong kernel. Select routiq-ai-coach/.venv, then restart the kernel.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
MODE = "OFFLINE_FOUNDATION"
print("Python:", platform.python_version())
print("Project:", project_root)
print("Mode:", MODE)

In [ ]:
# 2. Check installed packages; no network requests or installation.
import sys
from importlib.metadata import PackageNotFoundError, version
print("Selected Python:", sys.executable)
missing = []
for package in ("fastapi", "httpx", "pydantic", "crewai", "openai", "ipykernel"):
    try:
        print(package, version(package))
    except PackageNotFoundError:
        missing.append(package)
        print(package, "MISSING")
if missing:
    raise RuntimeError("Missing packages: " + ", ".join(missing) +
                       ". Select this module's .venv kernel and restart it.")
print("Packages ready; no agent or LLM client was started.")


In [ ]:
# 1. Find the existing module and verify its local kernel. No user/drive hardcoding.
from pathlib import Path
import platform
import sys

start = Path.cwd().resolve()
candidates = [Path(sys.prefix).resolve().parent, start, *start.parents,
              start / "routiq-ai-coach",
              start / "projects" / "routiq" / "routiq-ai-coach"]
project_root = next((p for p in candidates
                     if (p / "pyproject.toml").is_file()
                     and (p / "app" / "config" / "routiq_chatbot.json").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the Routiq AI Coach module and select its .venv kernel.")
project_root = project_root.resolve()
expected_venv = (project_root / ".venv").resolve()
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use the project's Python 3.12 kernel; supported range is >=3.10,<3.14.")
if Path(sys.prefix).resolve() != expected_venv:
    raise RuntimeError("Wrong kernel. Select routiq-ai-coach/.venv, then restart the kernel.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
MODE = "OFFLINE_FOUNDATION"
print("Python:", platform.python_version())
print("Project:", project_root)
print("Mode:", MODE)
# 3. Load the preserved project configuration. Never load or display secret values.
from app.config.loader import load_config

coach_config = load_config(project_root / "app" / "config" / "routiq_chatbot.json")
env_path = project_root / ".env"
print("Configuration:", coach_config.document.version, coach_config.document.status)
print("Proposed questions:", len(coach_config.sample_questions))
print("Module .env:", "present but not loaded" if env_path.is_file() else "not required in offline mode")
print("API key access: disabled. Product decisions and thresholds remain unresolved.")

In [ ]:
# 1. Find the existing module and verify its local kernel. No user/drive hardcoding.
from pathlib import Path
import platform
import sys

start = Path.cwd().resolve()
candidates = [Path(sys.prefix).resolve().parent, start, *start.parents,
              start / "routiq-ai-coach",
              start / "projects" / "routiq" / "routiq-ai-coach"]
project_root = next((p for p in candidates
                     if (p / "pyproject.toml").is_file()
                     and (p / "app" / "config" / "routiq_chatbot.json").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the Routiq AI Coach module and select its .venv kernel.")
project_root = project_root.resolve()
expected_venv = (project_root / ".venv").resolve()
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use the project's Python 3.12 kernel; supported range is >=3.10,<3.14.")
if Path(sys.prefix).resolve() != expected_venv:
    raise RuntimeError("Wrong kernel. Select routiq-ai-coach/.venv, then restart the kernel.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
MODE = "OFFLINE_FOUNDATION"
print("Python:", platform.python_version())
print("Project:", project_root)
print("Mode:", MODE)
# 4. Create an in-memory FastAPI test client, not a direct OpenAI client.
from fastapi.testclient import TestClient
from app.main import create_app
from app.api.chat import Principal, authenticated_principal
from app.schemas.chat import ChatRequest

notebook_app = create_app(project_root / "app" / "config" / "routiq_chatbot.json")
# Explicit test-only identity: affects this in-memory app instance only.
notebook_app.dependency_overrides[authenticated_principal] = lambda: Principal("NOTEBOOK_SYNTHETIC_USER")
client = TestClient(notebook_app)
context = {"screen": "PROGRESS", "habit_id": None}
with client:
    assert client.get("/health").json()["live_ai_enabled"] is False
    print("Readiness:", client.get("/ready").json())
print("Client ready for offline boundary testing; authentication is synthetic.")

In [ ]:
# 1. Find the existing module and verify its local kernel. No user/drive hardcoding.
from pathlib import Path
import platform
import sys

start = Path.cwd().resolve()
candidates = [Path(sys.prefix).resolve().parent, start, *start.parents,
              start / "routiq-ai-coach",
              start / "projects" / "routiq" / "routiq-ai-coach"]
project_root = next((p for p in candidates
                     if (p / "pyproject.toml").is_file()
                     and (p / "app" / "config" / "routiq_chatbot.json").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the Routiq AI Coach module and select its .venv kernel.")
project_root = project_root.resolve()
expected_venv = (project_root / ".venv").resolve()
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use the project's Python 3.12 kernel; supported range is >=3.10,<3.14.")
if Path(sys.prefix).resolve() != expected_venv:
    raise RuntimeError("Wrong kernel. Select routiq-ai-coach/.venv, then restart the kernel.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
MODE = "OFFLINE_FOUNDATION"
print("Python:", platform.python_version())
print("Project:", project_root)
print("Mode:", MODE)
# 1. Find the existing module and verify its local kernel. No user/drive hardcoding.
from pathlib import Path
import platform
import sys

start = Path.cwd().resolve()
candidates = [Path(sys.prefix).resolve().parent, start, *start.parents,
              start / "routiq-ai-coach",
              start / "projects" / "routiq" / "routiq-ai-coach"]
project_root = next((p for p in candidates
                     if (p / "pyproject.toml").is_file()
                     and (p / "app" / "config" / "routiq_chatbot.json").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the Routiq AI Coach module and select its .venv kernel.")
project_root = project_root.resolve()
expected_venv = (project_root / ".venv").resolve()
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use the project's Python 3.12 kernel; supported range is >=3.10,<3.14.")
if Path(sys.prefix).resolve() != expected_venv:
    raise RuntimeError("Wrong kernel. Select routiq-ai-coach/.venv, then restart the kernel.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
MODE = "OFFLINE_FOUNDATION"
print("Python:", platform.python_version())
print("Project:", project_root)
print("Mode:", MODE)
# 3. Load the preserved project configuration. Never load or display secret values.
from app.config.loader import load_config

coach_config = load_config(project_root / "app" / "config" / "routiq_chatbot.json")
env_path = project_root / ".env"
print("Configuration:", coach_config.document.version, coach_config.document.status)
print("Proposed questions:", len(coach_config.sample_questions))
print("Module .env:", "present but not loaded" if env_path.is_file() else "not required in offline mode")
print("API key access: disabled. Product decisions and thresholds remain unresolved.")
# 1. Find the existing module and verify its local kernel. No user/drive hardcoding.
from pathlib import Path
import platform
import sys

start = Path.cwd().resolve()
candidates = [Path(sys.prefix).resolve().parent, start, *start.parents,
              start / "routiq-ai-coach",
              start / "projects" / "routiq" / "routiq-ai-coach"]
project_root = next((p for p in candidates
                     if (p / "pyproject.toml").is_file()
                     and (p / "app" / "config" / "routiq_chatbot.json").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the Routiq AI Coach module and select its .venv kernel.")
project_root = project_root.resolve()
expected_venv = (project_root / ".venv").resolve()
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use the project's Python 3.12 kernel; supported range is >=3.10,<3.14.")
if Path(sys.prefix).resolve() != expected_venv:
    raise RuntimeError("Wrong kernel. Select routiq-ai-coach/.venv, then restart the kernel.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
MODE = "OFFLINE_FOUNDATION"
print("Python:", platform.python_version())
print("Project:", project_root)
print("Mode:", MODE)
# 4. Create an in-memory FastAPI test client, not a direct OpenAI client.
from fastapi.testclient import TestClient
from app.main import create_app
from app.api.chat import Principal, authenticated_principal
from app.schemas.chat import ChatRequest

notebook_app = create_app(project_root / "app" / "config" / "routiq_chatbot.json")
# Explicit test-only identity: affects this in-memory app instance only.
notebook_app.dependency_overrides[authenticated_principal] = lambda: Principal("NOTEBOOK_SYNTHETIC_USER")
client = TestClient(notebook_app)
context = {"screen": "PROGRESS", "habit_id": None}
with client:
    assert client.get("/health").json()["live_ai_enabled"] is False
    print("Readiness:", client.get("/ready").json())
print("Client ready for offline boundary testing; authentication is synthetic.")
# 5. Interactive Coach: offline by default; /live requires explicit consent.
from app.config.settings import LiveSettings
from app.services.chat_service import LiveCoachService
from pydantic import ValidationError

local_history = []
print("Routiq ready in OFFLINE_FOUNDATION mode.")
print("Use /live for live coaching, /help for commands, or exit to stop.")
with client:
    while True:
        try:
            message = input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("Chat stopped.")
            break
        if message.lower() in {"exit", "quit", "/exit"}:
            break
        if not message:
            print("Enter a message, or exit to stop.")
            continue
        if message == "/help":
            print("/live: enable live coaching; /offline: disable live coaching; "
                  "/samples: question examples; /status: readiness; "
                  "/reset: clear local history; exit: stop.")
            continue
        if message == "/live":
            print("Live mode sends your next questions to OpenAI and may incur API charges.")
            print("Do not enter passwords, API keys or sensitive records in chat.")
            try:
                choice = input("Type yes to enable paid OpenAI replies, or press Enter to cancel: ").strip()
            except (KeyboardInterrupt, EOFError):
                print("Live mode cancelled.")
                continue
            if " ".join(choice.casefold().split()) not in {"enable live", "yes"}:
                print("Live mode cancelled; existing mode unchanged.")
                continue
            try:
                live_settings = LiveSettings.load(project_root)
            except ValueError as error:
                print("Live mode not enabled:", str(error))
                continue
            notebook_app.state.coach_service = LiveCoachService(live_settings)
            MODE = "LIVE_GENERAL_COACH"
            print("Live general coaching enabled. Personal habit records are not connected.")
            print("Each question is independent; previous messages are not sent to OpenAI.")
            continue
        if message == "/offline":
            notebook_app.state.coach_service = None
            MODE = "OFFLINE_FOUNDATION"
            print("Live coaching disabled.")
            continue
        if message == "/samples":
            print("Live example: Help me start a reading habit.")
            print("Live example: Suggest an optional experiment for a morning routine.")
            print("Configuration examples below may require habit data that is not connected:")
            for question in coach_config.sample_questions[:5]:
                print(f"{question.id}: {question.text} (proposed: {question.intent_id})")
            continue
        if message == "/status":
            print("Mode:", MODE)
            print(client.get("/ready").json())
            continue
        if message == "/reset":
            local_history.clear()
            print("Local history cleared. No stored conversation exists.")
            continue
        try:
            request = ChatRequest(message=message, conversation_id=None, context=context)
        except ValidationError:
            print("Message or screen context is invalid. Use 1-4000 characters and a supported screen.")
            continue
        response = client.post("/api/v1/ai/chat", json=request.model_dump())
        if response.status_code == 200:
            reply = response.json()["message"]
        elif response.status_code == 503:
            reply = response.json()["message"]
        elif response.status_code == 401:
            reply = "Verified authentication is not integrated."
        else:
            reply = "The coach could not handle this request. Check /status and try again."
        local_history.append({"user": message, "foundation_response": reply})
        print("Routiq:", reply)
notebook_app.state.coach_service = None
MODE = "OFFLINE_FOUNDATION"
print("Session ended; live mode disabled. Rerun this cell for fresh local history.")
